In [ ]:
# Define coordinates as a list of [lon, lat]
coordinates_list = [
    # [lon, lat], # Add more coordinates as needed
    [91.2734, 29.7904], # 拉萨
    [91.9113, 29.1816], # 山南
    [92.6035, 28.9443], # 加查
    [94.2605, 29.7733], # 尼西村
    [94.3610, 29.6483], # 林芝
    [100.1120, 25.6540], # 苍山
    [100.7173, 22.1298], # 西双版纳
    [101.2663, 24.3681], # 景东
    [102.2691, 27.9628], # 凉山
    [104.6592, 29.2588], # 自贡
    [109.0416, 24.3102], # 柳州 云太多
    [115.8977, 33.0074], # 阜阳 云太多
    [116.3372, 31.3563], # 南岳
    [118.1656, 30.1375], # 黄山
    [77.612683, 22.125727],
    [108.605387, 29.199306],
    [88.962192, 30.977011],
    [102.183343, 44.363339],
    [32.418515, 24.869215],
    [17.578084, -2.190697],
    [-41.739767, -5.546400],
    [-85.876210, 34.033028],
    [-101.337808, 37.259190],
    [-107.275097, 51.411700],
    [49.555324, 52.340537],
]

# Image Size in pixels
pixel_size = 10240

image_collection_id = "COPERNICUS/S2_HARMONIZED"
bands_to_export = [
    "B2", # Blue
    "B3", # Green
    "B4", # Red
    "B8", # NIR
    "B11",# SWIR1
    "B12" # SWIR2
]
start_date = "2015-01-01"
end_date = "2026-12-31"
use_csplus = True
csplus_thresh = 0.8
cloud_filter_percentage = 20
local_output_dir = "../data/sentinel-new"
export_scale = 10
export_crs = "EPSG:4326"
skip_existing = True
max_tile_dim = 1024


In [ ]:
import ee
from ee.geometry import Geometry
from ee.imagecollection import ImageCollection
from ee.image import Image
from ee.filter import Filter
from ee.join import Join
from pathlib import Path
import geemap

try:
    ee.Initialize(project="ee-yangluhao990714")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="ee-yangluhao990714")

In [ ]:
for coords in coordinates_list:
    lon, lat = coords[0], coords[1]

    # Calculate buffer size in meters to get desired pixel dimensions
    buffer_size = (pixel_size * export_scale) / 2
    point = Geometry.Point([lon, lat])
    bbox = point.buffer(buffer_size).bounds()

    image_collection: ImageCollection = (ImageCollection(image_collection_id)
                        .filterBounds(bbox)
                        .filterDate(start_date, end_date))

    if cloud_filter_percentage is not None:
        image_collection = image_collection.filter(
            Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_filter_percentage)
        )

    if use_csplus:
        cs = ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
        join_filter = Filter.equals(leftField="system:index", rightField="system:index")
        join = Join.saveFirst(matchKey="csplus_mask")
        joined = join.apply(image_collection, cs, join_filter)
        def add_mask(img):
            img = ee.Image(img)
            cs_img = ee.Image(img.get("csplus_mask"))
            return img.updateMask(cs_img.select("cs").gt(csplus_thresh))
        image_collection = ee.ImageCollection(joined.map(add_mask))

    output_dir = Path(local_output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    for band_name in bands_to_export:
        time_series_cube = image_collection.select(band_name).toBands()

        collection_name_for_file = image_collection_id.replace("/", "_")
        file_name = f"{collection_name_for_file}_{band_name}_lon{lon:.4f}_lat{lat:.4f}"
        output_path = output_dir / f"{file_name}.tif"

        if skip_existing and output_path.exists():
            print(f"Skip existing: {output_path}")
            continue

        print(f"Downloading {file_name} -> {output_path}")
        try:
            geemap.download_ee_image(
                image=time_series_cube.toFloat(),
                filename=str(output_path),
                region=bbox,
                scale=export_scale,
                crs=export_crs,
                overwrite=not skip_existing,
                max_tile_dim=max_tile_dim,
            )
        except Exception as exc:
            print(f"Failed: {file_name}: {exc}")
